In [26]:
# ============================================================
# FLEX + BISON
# Program to recognize a valid variable
# Variable must start with a letter,
# followed by any number of letters or digits
# ============================================================

# Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# 1. Create FLEX file : valvar.l
# ============================================================

lexer_code = r'''
%{
#include "valvar.tab.h"
%}

%%
[a-zA-Z]    { return LET; }
[0-9]       { return DIG; }
[ \t]+      { /* ignore spaces */ }
\n          { return 0; }
.           { return yytext[0]; }
%%

int yywrap()
{
    return 1;
}
'''

with open("valvar.l", "w") as f:
    f.write(lexer_code)


# ============================================================
# 2. Create BISON file : valvar.y
# ============================================================

parser_code = r'''
%{
#include <stdio.h>
#include <stdlib.h>

int yylex();
int yyerror(const char *s);
%}

%token LET DIG

%%

variable:
      var
      ;

var:
      var DIG
    | var LET
    | LET
    ;

%%

int main()
{
    printf("Enter the variable:\n");

    if (yyparse() == 0)
    {
        printf("Valid variable\n");
    }

    return 0;
}

int yyerror(const char *s)
{
    printf("Invalid variable\n");
    exit(0);
}
'''

with open("valvar.y", "w") as f:
    f.write(parser_code)


# ============================================================
# 3. Generate BISON parser
# ============================================================

!bison -d valvar.y


# ============================================================
# 4. Generate FLEX scanner
# ============================================================

!flex valvar.l


# ============================================================
# 5. Compile
# ============================================================

!gcc lex.yy.c valvar.tab.c -o valvar -lfl


# ============================================================
# 6. Test 1 - Valid variable
# ============================================================

print("\n========== TEST 1 ==========\n")

with open("input.txt", "w") as f:
    f.write("add\n")

!./valvar < input.txt


# ============================================================
# 7. Test 2 - Valid variable with digits
# ============================================================

print("\n========== TEST 2 ==========\n")

with open("input.txt", "w") as f:
    f.write("add1\n")

!./valvar < input.txt


# ============================================================
# 8. Test 3 - Invalid variable
# ============================================================

print("\n========== TEST 3 ==========\n")

with open("input.txt", "w") as f:
    f.write("1add\n")

!./valvar < input.txt

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

========== TEST 1 ==========

Enter the variable:
Valid variable

========== TEST 2 ==========

Enter the variable:
Valid variable

========== TEST 3 ==========

Enter the variable:
Invalid variable
